In [1]:
!git clone https://github.com/iesxz-c/Final.git
%cd Final

Cloning into 'Final'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 196 (delta 86), reused 162 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 183.36 KiB | 1.38 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/Final


In [2]:
!nvidia-smi  # confirm Tesla T4 + CUDA


Wed Sep 16 11:43:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!mkdir -p data
!cp /content/drive/MyDrive/phase2c_remaining_702.json data/phase2c_remaining_702.json

In [5]:
!python -c "import json; x=json.load(open('data/phase2c_remaining_702.json')); print('Manifest videos:',len(x['videos']))"

Manifest videos: 702


In [6]:
!python scripts/run_resumable_ucf.py \
  --input-manifest data/phase2c_remaining_702.json \
  --output-dir /content/drive/MyDrive/Crime_CCTV_Project/Phase2/phase2c_ucf_batches \
  --device cuda

TOTAL: 702
COMPLETED BEFORE START: 600
REMAINING: 102
CURRENT BATCH: batch_013 (50 videos)
Event model: OPear/videomae-large-finetuned-UCF-Crime (bc5f1c1058158a05f8710de2b7c7c372fe69062b) on cuda (16 frames @ 8.0fps, top-5)
[1/50] anomaly/Stealing/Stealing047_x264.mp4
  22 windows (video 30.00fps)
[2/50] anomaly/Stealing/Stealing048_x264.mp4
  119 windows (video 30.00fps)
[3/50] anomaly/Stealing/Stealing049_x264.mp4
  24 windows (video 30.00fps)
[4/50] anomaly/Stealing/Stealing050_x264.mp4
  53 windows (video 30.00fps)
[5/50] anomaly/Stealing/Stealing051_x264.mp4
  53 windows (video 30.00fps)
[6/50] anomaly/Stealing/Stealing052_x264.mp4
  53 windows (video 30.00fps)
[7/50] anomaly/Stealing/Stealing053_x264.mp4
  205 windows (video 30.00fps)
[8/50] anomaly/Stealing/Stealing054_x264.mp4
  42 windows (video 30.00fps)
[9/50] anomaly/Stealing/Stealing055_x264.mp4
  26 windows (video 30.00fps)
[10/50] anomaly/Stealing/Stealing057_x264.mp4
  48 windows (video 30.00fps)
[11/50] anomaly/Stealin

In [7]:
import torch, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(2000, 2000, device=device)

print("Keeping GPU warm... interrupt (stop button) when you're done.")
try:
    while True:
        x = x @ x
        torch.cuda.synchronize()
        time.sleep(5)   # throttle so it's not pointlessly burning compute
except KeyboardInterrupt:
    print("Stopped.")

Keeping GPU warm... interrupt (stop button) when you're done.
Stopped.


In [8]:
!python scripts/run_resumable_ucf.py \
  --input-manifest data/phase2c_remaining_702.json \
  --output-dir /content/drive/MyDrive/Crime_CCTV_Project/Phase2/phase2c_ucf_batches \
  --device cuda \
  --merge

merged 702 videos, 44716 observations -> /content/Final/data/evidence/phase2c_ucf_remaining_702


In [10]:
import json
from pathlib import Path
from collections import Counter

root = Path("data/evidence/phase2c_ucf_remaining_702")

manifest = json.load(open(root / "manifest.json"))
videos_raw = json.load(open(root / "videos.json"))
events = json.load(open(root / "ucf_events.json"))

videos = videos_raw["videos"] if isinstance(videos_raw, dict) else videos_raw

video_ids = [v["video_id"] for v in videos]
event_ids = [e["observation_id"] for e in events]
event_video_ids = {e["video_id"] for e in events}

print("=== MERGED 702 VALIDATION ===")
print("Videos:", len(videos))
print("Unique video IDs:", len(set(video_ids)))
print("Observations:", len(events))
print("Unique observation IDs:", len(set(event_ids)))
print("Event video IDs:", len(event_video_ids))

print("\n=== TRACEABILITY ===")
print("Videos with no events:", len(set(video_ids) - event_video_ids))
print("Events referencing unknown videos:", len(event_video_ids - set(video_ids)))

print("\n=== MODEL ===")
print("Models:", Counter(e.get("model_name") for e in events))

print("\n=== SCHEMA ===")
print("Video schema versions:", Counter(v.get("schema_version") for v in videos))
print("Event schema versions:", Counter(e.get("schema_version") for e in events))

print("\n=== RESULT ===")
checks = [
    len(videos) == 702,
    len(set(video_ids)) == 702,
    len(events) == 44716,
    len(set(event_ids)) == 44716,
    len(set(video_ids) - event_video_ids) == 0,
    len(event_video_ids - set(video_ids)) == 0,
]

print("PASS" if all(checks) else "FAIL")

=== MERGED 702 VALIDATION ===
Videos: 702
Unique video IDs: 702
Observations: 44716
Unique observation IDs: 44716
Event video IDs: 702

=== TRACEABILITY ===
Videos with no events: 0
Events referencing unknown videos: 0

=== MODEL ===
Models: Counter({'OPear/videomae-large-finetuned-UCF-Crime': 44716})

=== SCHEMA ===
Video schema versions: Counter({'phase2a/v1': 702})
Event schema versions: Counter({'phase2a/v1': 44716})

=== RESULT ===
PASS


In [11]:
!mkdir -p /content/drive/MyDrive/Crime_CCTV_Project/Phase2/phase2c_ucf_remaining_702
!cp data/evidence/phase2c_ucf_remaining_702/*.json \
   /content/drive/MyDrive/Crime_CCTV_Project/Phase2/phase2c_ucf_remaining_702/

In [12]:
!ls -lh /content/drive/MyDrive/Crime_CCTV_Project/Phase2/phase2c_ucf_remaining_702/

total 49M
-rw------- 1 root root  628 Sep 16 13:05 manifest.json
-rw------- 1 root root  48M Sep 16 13:05 ucf_events.json
-rw------- 1 root root 323K Sep 16 13:05 videos.json
